# Neo4j GraphRAG + LangChain

[`neo4j-graphrag`](https://neo4j.com/docs/neo4j-graphrag-python/current/) is Neo4j's official
retrieval library. It provides **retrievers** - objects that turn a question into results from your
graph using vector search, full-text search, graph traversal, or a combination of all three.

In this notebook, we'll wrap those retrievers as LangChain tools and give them to an agent that
chooses between them.

Retrievers are plain Python objects with a `search()` method, so they drop into LangChain the same
way any other function does - via the `@tool` decorator.

In [2]:
!pip install --quiet --upgrade langchain langchain-openai neo4j-graphrag neo4j openai


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import json
import math
import os
from getpass import getpass

import neo4j
from neo4j import GraphDatabase

from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.retrievers import (
    VectorRetriever,
    VectorCypherRetriever,
    HybridRetriever,
    HybridCypherRetriever,
)
from neo4j_graphrag.types import RetrieverResultItem

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

We'll use OpenAI as the LLM provider, specifically **GPT-5.4-mini**. The same API key is used for the
embeddings later on.

In [2]:
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key")

model = ChatOpenAI(model="gpt-5.4-mini")

## The graph

We'll use the companies database from the Neo4j demo server. Articles are split into `Chunk` nodes
that carry the text and its embeddings, and linked to the organizations they mention:

```
(Article)-[:HAS_CHUNK]->(Chunk)
(Article)-[:MENTIONS]->(Organization)
(Organization)-[:HAS_COMPETITOR|HAS_INVESTOR|HAS_SUPPLIER]->(Organization)
```

That shape is what makes graph retrieval worthwhile. A vector search finds a *chunk* - but the
article it belongs to, its date, its sentiment, and the companies it discusses are all one hop away.

In [3]:
NEO4J_URI = "neo4j+s://demo.neo4jlabs.com"
NEO4J_DATABASE = "companies"
NEO4J_USERNAME = "companies"
NEO4J_PASSWORD = "companies"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
print("Connected to", NEO4J_DATABASE)

Connected to companies


### Cypher version

`neo4j-graphrag` generates its vector searches using the Cypher 25 `SEARCH` clause. The demo
database understands Cypher 25 but still defaults to Cypher 5, so we prefix the generated queries
to select the newer language version.

Skip this cell if your own database already defaults to Cypher 25.

In [4]:
if not getattr(neo4j.Driver.execute_query, "_cypher25_patch", False):
    _execute_query = neo4j.Driver.execute_query

    def _execute_query_cypher25(self, query, *args, **kwargs):
        if isinstance(query, str) and "VECTOR INDEX" in query.upper() \
                and not query.lstrip().upper().startswith("CYPHER"):
            query = "CYPHER 25 " + query
        return _execute_query(self, query, *args, **kwargs)

    _execute_query_cypher25._cypher25_patch = True
    neo4j.Driver.execute_query = _execute_query_cypher25

print("Generated vector queries will run as Cypher 25.")

Generated vector queries will run as Cypher 25.


## Pair the embedding model with the index

A vector index stores embeddings produced by one specific model. Querying it with a different model
still returns results, but they're meaningless - two models place the same text at different points
in space, so the similarity scores are noise. Nothing raises an exception when this happens.

Let's look at what indexes exist on `Chunk`.

In [5]:
indexes, _, _ = driver.execute_query(
    """
    SHOW INDEXES YIELD name, type, labelsOrTypes, properties, options
    WHERE type IN ['VECTOR', 'FULLTEXT'] AND labelsOrTypes = ['Chunk']
    RETURN name, type, properties,
           options.indexConfig['vector.dimensions'] AS dimensions
    ORDER BY type, name
    """,
    database_=NEO4J_DATABASE,
)

for record in indexes:
    row = record.data()
    dims = f"{row['dimensions']} dims" if row["dimensions"] else "-"
    print(f"  {row['name']:<18} {row['type']:<9} {str(row['properties']):<26} {dims}")

  news_fulltext      FULLTEXT  ['text']                   -
  news               VECTOR    ['embedding']              1536 dims
  news_google        VECTOR    ['embedding_google']       768 dims
  news_google_004    VECTOR    ['embedding_google_004']   768 dims
  news_sbert         VECTOR    ['embedding_sbert']        384 dims


The demo graph stores several embeddings of the same chunks, one per model, so you can use whichever
provider you already have access to.

We'll use **`news`**, whose 1536 dimensions correspond to OpenAI's `text-embedding-ada-002` - the
default model for `neo4j-graphrag`'s `OpenAIEmbeddings`. Since we're already using OpenAI for the
LLM, no extra credentials are needed.

**`news_fulltext`** indexes the `text` property on those same `Chunk` nodes. Hybrid retrieval needs
a vector index and a full-text index over the same nodes, which is what makes that section possible.

In [6]:
VECTOR_INDEX = "news"
FULLTEXT_INDEX = "news_fulltext"

embedder = OpenAIEmbeddings()

print(f"{len(embedder.embed_query('test'))} dimensions")

1536 dimensions


To confirm the pairing, re-embed a chunk that already has a stored vector and compare the two. An
exact model match scores close to 1.0; anything much lower means the index was built with a
different model, and you should pick another index or another embedder.

Worth running whenever you point this code at a new database.

In [7]:
def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))


sample, _, _ = driver.execute_query(
    "MATCH (c:Chunk) WHERE c.embedding IS NOT NULL "
    "RETURN c.text AS text, c.embedding AS stored LIMIT 1",
    database_=NEO4J_DATABASE,
)

score = cosine(embedder.embed_query(sample[0]["text"]), sample[0]["stored"])
print(f"Self-similarity: {score:.3f}",
      "- models match" if score > 0.95 else "- MISMATCH, pick a different index or embedder")

Self-similarity: 1.000 - models match


## VectorRetriever

The simplest retriever. Give it a driver, an index name, and an embedder; it embeds the question,
searches the index, and returns matching nodes.

Set `return_properties` to control what comes back - without it you get whole nodes, embeddings
included, which is thousands of floats per result heading straight into your prompt.

In [8]:
vector_retriever = VectorRetriever(
    driver=driver,
    index_name=VECTOR_INDEX,
    embedder=embedder,
    return_properties=["text"],
    neo4j_database=NEO4J_DATABASE,
)

QUERY = "renewable energy investment"

for item in vector_retriever.search(query_text=QUERY, top_k=3).items:
    print(f"  {item.content[:160]}...\n")

  {'text': "The publisher has been monitoring the renewable energy investment market and it is poised to grow by $168.43 bn during 2022-2026 decelerating at a CAG...

  {'text': ' and be a leader in renewable energy.'}...

  {'text': ' and be a leader in renewable energy.'}...



This is ordinary RAG: it finds relevant text and stops. The retriever knows a chunk matched, but not
which article it came from, when it was published, or which companies it discusses.

## VectorCypherRetriever

This is the retriever that separates GraphRAG from RAG.

It runs the same vector search, then executes a `retrieval_query` starting from each match. Two
variables are in scope for that query: `node`, the node found by the vector search, and `score`, its
similarity. From there you can traverse anywhere in the graph.

Below: chunk → article (title, date, sentiment) → organizations mentioned → their competitors. The
model receives the text *and* the surrounding facts in a single round trip.

A `result_formatter` shapes each returned record. Without one you get the driver's default record
repr, which is awkward to read and awkward to hand to a model.

In [9]:
RETRIEVAL_QUERY = """
WITH node AS chunk, score
MATCH (article:Article)-[:HAS_CHUNK]->(chunk)
OPTIONAL MATCH (article)-[:MENTIONS]->(org:Organization)
OPTIONAL MATCH (org)-[:HAS_COMPETITOR]->(rival:Organization)
RETURN chunk.text             AS text,
       article.id             AS article_id,
       article.title          AS title,
       toString(article.date) AS date,
       article.sentiment      AS sentiment,
       collect(DISTINCT org.name)[..5]   AS companies,
       collect(DISTINCT rival.name)[..5] AS competitors,
       score
ORDER BY score DESC
"""


def format_record(record) -> RetrieverResultItem:
    """Shape each result into compact JSON for the model."""
    return RetrieverResultItem(
        content=json.dumps({
            "text": (record.get("text") or "")[:600],
            "article_id": record.get("article_id"),
            "title": record.get("title"),
            "date": record.get("date"),
            "sentiment": record.get("sentiment"),
            "companies": record.get("companies"),
            "competitors": record.get("competitors"),
        }, default=str),
        metadata={"score": record.get("score")},
    )


graph_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=VECTOR_INDEX,
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=format_record,
    neo4j_database=NEO4J_DATABASE,
)

for item in graph_retriever.search(query_text=QUERY, top_k=2).items:
    row = json.loads(item.content)
    print(f"  {row['title']}  ({row['date']}, sentiment {row['sentiment']})")
    print(f"    companies: {row['companies']}")
    print(f"    competitors: {row['competitors']}")
    print(f"    {row['text'][:120]}...\n")

  BYD is overtaking Tesla, but its EV dream skips the US for now  (2023-06-15T09:13:53Z, sentiment 0.0)
    companies: ['Tesla', 'BYD Auto']
    competitors: []
     and be a leader in renewable energy....



Same query and same index as before, but every result now carries an article ID, a date, a stored
sentiment score, and the companies involved.

## HybridRetriever

Vector search is strong on meaning and weak on exact strings. Ask about a specific company, ticker,
or product and it tends to return material on the right *topic* while missing the document that
names it outright. Full-text search has the opposite profile.

`HybridRetriever` runs both and merges the rankings.

In [10]:
hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name=VECTOR_INDEX,
    fulltext_index_name=FULLTEXT_INDEX,
    embedder=embedder,
    return_properties=["text"],
    neo4j_database=NEO4J_DATABASE,
)

NAMED_QUERY = "Nvidia data center GPU demand"

print("--- vector only ---")
for item in vector_retriever.search(query_text=NAMED_QUERY, top_k=3).items:
    print(f"  {item.content[:120]}...")

print("\n--- hybrid ---")
for item in hybrid_retriever.search(query_text=NAMED_QUERY, top_k=3).items:
    print(f"  {item.content[:120]}...")

--- vector only ---
  {'text': '. Traditional GPU uses include professional visualization applications that require realistic rendering, inclu...
  {'text': '. Hyperscale cloud vendors have leveraged GPUs in training neural networks for uses such as image and speech r...
  {'text': '(Reuters) - Nvidia Corp <NVDA.O> on Monday laid out a multi-year plan to create a new kind of chip for data ce...

--- hybrid ---
  {'text': 'Supermicro Founder and CEO Charles Liang Will be Joined by Jensen Huang, NVIDIA CEO, and other Industry Lumina...
  {'text': '. Traditional GPU uses include professional visualization applications that require realistic rendering, inclu...
  {'text': 'Supermicro Founder and CEO Charles Liang Will be Joined by Jensen Huang, NVIDIA CEO, and other Industry Lumina...


The hybrid results surface chunks naming the company directly, while vector-only results drift toward
the general theme. The more distinctive the term - proper nouns, tickers, product codes - the wider
the gap.

`HybridCypherRetriever` combines both ideas: hybrid search plus a `retrieval_query`. It's the one to
reach for in practice, and the one we'll give the agent.

In [11]:
hybrid_graph_retriever = HybridCypherRetriever(
    driver=driver,
    vector_index_name=VECTOR_INDEX,
    fulltext_index_name=FULLTEXT_INDEX,
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=format_record,
    neo4j_database=NEO4J_DATABASE,
)

row = json.loads(hybrid_graph_retriever.search(query_text=NAMED_QUERY, top_k=1).items[0].content)
print(json.dumps(row, indent=2)[:600])

{
  "text": ". Traditional GPU uses include professional visualization applications that require realistic rendering, including computer-aided design, video editing, and special effects. Nvidia has experienced success in focusing its GPUs in burgeoning markets such as artificial intelligence (deep learning) and self-driving vehicles. Hyperscale cloud vendors have leveraged GPUs in training neural networks for uses such as image and speech recognition, large language models (ChatGPT), and other forms of generative AI.\u201d\n\u2014Brian Colello, director of technology equity research\n\u201cWe 


# Retrievers as LangChain tools

The `@tool` decorator turns any function into a tool the agent can invoke. LangChain builds the tool
schema from the function name, type hints, and docstring - so **the docstring is the interface**.
It's how the model decides which retriever fits the question in front of it.

Here are three retrieval strategies with genuinely different strengths, and an agent that chooses
between them.

In [12]:
@tool
def search_news(question: str) -> str:
    """Search news article text by meaning. Best for broad themes and open-ended questions,
    for example "what are the concerns around AI regulation"."""
    result = vector_retriever.search(query_text=question, top_k=5)
    return json.dumps([item.content for item in result.items], indent=2)


@tool
def search_news_with_context(question: str) -> str:
    """Search news by meaning and return graph context with each result: the source article,
    its date and sentiment, the companies mentioned, and their competitors. Use this whenever
    the answer needs citations or company relationships."""
    result = hybrid_graph_retriever.search(query_text=question, top_k=5)
    return json.dumps([json.loads(item.content) for item in result.items], indent=2)


@tool
def find_named_entity(question: str) -> str:
    """Search news for an exact company name, ticker, or product that appears literally in the
    text. Use this when the question centres on a specific named thing rather than a theme."""
    result = hybrid_retriever.search(query_text=question, top_k=5)
    return json.dumps([item.content for item in result.items], indent=2)


system_prompt = """
You answer questions about companies using a Neo4j news graph.

Choose the retrieval tool that fits the question:
- search_news for broad themes and open-ended topics.
- search_news_with_context when the answer needs citations, dates, sentiment, or company
  relationships. Prefer this one for anything analytical.
- find_named_entity when the question is about a specific named company, ticker, or product.

Cite the article title and date when the tool provides them. If the retrieved passages do not
answer the question, say so rather than filling the gap from your own knowledge.
"""

agent = create_agent(
    model,
    [search_news, search_news_with_context, find_named_entity],
    system_prompt=system_prompt,
)

Let's test it!

In [13]:
prompt = "Which companies are expanding in renewable energy, and who competes with them?"

async for event in agent.astream(
    {"messages": [{"role": "user", "content": prompt}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Which companies are expanding in renewable energy, and who competes with them?
================================== Ai Message ==================================
Tool Calls:
  search_news_with_context (call_RRCb6cO4xxZydnzWVi5ezSMt)
 Call ID: call_RRCb6cO4xxZydnzWVi5ezSMt
  Args:
    question: Which companies are expanding in renewable energy, and who competes with them? Include article titles, dates, and competitor relationships.
================================= Tool Message =================================
Name: search_news_with_context

[
  {
    "text": "2019 MAR 11 (NewsRx) -- By a News Reporter-Staff News Editor at Energy Daily News -- Current study results on Energy - Modern Power Systems and Clean Energy have been published. According to news reporting originating in Cambridge, Massachusetts, by NewsRx journalists, research stated, \u201cInvestment for renewables has been growing rapidly since the

In [14]:
prompt = "What are the recent trends in cloud computing?"

async for event in agent.astream(
    {"messages": [{"role": "user", "content": prompt}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What are the recent trends in cloud computing?
================================== Ai Message ==================================
Tool Calls:
  search_news_with_context (call_ssTrfgvCwUguX9zomtvPzfZi)
 Call ID: call_ssTrfgvCwUguX9zomtvPzfZi
  Args:
    question: recent trends in cloud computing, including major themes, dates, and any company relationships or sentiment in news coverage
================================= Tool Message =================================
Name: search_news_with_context

[
  {
    "text": "According to Ovum\u2019s 2014 Trends-to-Watch: Cloud Services* report, which explains that adoption of cloud services has moved from a nice-to-have component of corporate IT to a strategic imperative.\nMore on This\nEuropean Broadband Connectivity to Soar Over the Next Five Years, Ovum Says\nMobile Data Services Rise in Africa, but Continent Lags in Connectivity\nJohn Madden, IT Services Practice 

Watch which tool the agent reaches for in each run. Nothing routes those calls but the docstrings -
no keyword matching, no conditional logic. Rewriting a description is the fastest way to correct a
retriever that's being over- or under-used.

## Comparing strategies side by side

Tool choice is the agent's problem. Knowing which retriever suits *your* data is yours. Run all three
over one question and read the results together.

In [15]:
COMPARISON_QUERY = "semiconductor manufacturing capacity"

for label, retriever in [
    ("vector", vector_retriever),
    ("hybrid", hybrid_retriever),
    ("hybrid + graph", hybrid_graph_retriever),
]:
    print(f"\n=== {label} ===")
    for item in retriever.search(query_text=COMPARISON_QUERY, top_k=2).items:
        try:
            row = json.loads(item.content)
            print(f"  [{row['title']}] {row['text'][:130].strip()}...")
        except (json.JSONDecodeError, KeyError):
            print(f"  {item.content[:150].strip()}...")


=== vector ===
  {'text': ', therefore, demand for semiconductor production equipment – is extremely sensitive to market conditions.\nWhen they deteriorate, companies...
  {'text': 'Demand for semiconductors and the equipment used in their production has been sky high in recent years, an up-and-up trend industry forecast...

=== hybrid ===
  {'text': ', therefore, demand for semiconductor production equipment – is extremely sensitive to market conditions.\nWhen they deteriorate, companies...
  {'text': "Following a DIGITIMES Asia report in mid-May indicating that Chinese foundry SMIC has removed its 14nm FinFET offering from its website, rec...

=== hybrid + graph ===
  [Semiconductor cycle shows signs of peaking] , therefore, demand for semiconductor production equipment – is extremely sensitive to market conditions.
When they deteriorate, c...
  [SMIC subsidiary continues to fulfill Chinese demands for FinFET capabilities] Following a DIGITIMES Asia report in mid-May indicating that

---

## Summary

In this notebook, we built a news research agent using `neo4j-graphrag` with LangChain:

1. **Index pairing** - matched the embedding model to the vector index that was built with it, and confirmed it by re-embedding a stored chunk
2. **VectorRetriever** - semantic search over article text
3. **VectorCypherRetriever** - the same search, continued into the graph to bring back articles, dates, sentiment, and company relationships
4. **HybridRetriever** - vector and full-text combined, recovering exact matches on names that embeddings miss
5. **Retrievers as tools** - three strategies exposed with the `@tool` decorator, selected by the agent per question

Retrievers bundle an index, an embedding model, and optionally a traversal query behind a single
`search()` method - so swapping one for another changes retrieval behaviour without touching the
agent around it.

In [16]:
driver.close()